
# Roxy notebook example: Hydrophobicity, polarity, and amphipathicity descriptors

This notebook is a **reference implementation example** for the **hydrophobicity, polarity, and amphipathicity descriptor family** in Roxy.

These descriptors summarize both **global** and **local** sequence behavior related to:

- hydrophobicity
- polarity
- hydrophilic/hydrophobic balance
- local physicochemical contrast
- simple amphipathic tendencies inferred from sequence

They are useful because many biologically relevant peptides and proteins are characterized not only by their overall composition, but also by the presence of **locally segregated hydrophobic and polar patches**.

## Covered outputs

This notebook implements examples such as:

- mean hydrophobicity
- mean polarity
- hydrophobicity and polarity standard deviation
- hydrophobic / hydrophilic fractions
- polar / nonpolar fractions
- hydrophobicity amplitude
- polarity amplitude
- sliding-window hydrophobicity and polarity profiles
- local hydrophobic patch burden
- local polar patch burden
- simple amphipathicity contrast proxies
- hydrophobic-polar balance scores
- terminal hydrophobicity and polarity asymmetry
- class-style implementation for later migration into Roxy


In [1]:

from collections import Counter

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "hpa_1",
            "hpa_2",
            "hpa_3",
            "hpa_4",
            "hpa_5",
            "hpa_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,hpa_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,hpa_2,GGGGGGGGGGGGGGG,B
2,hpa_3,KRRKRRKRRKRRDDDDEE,A
3,hpa_4,ACDEFGHIKLMNPQRSTVWY,B
4,hpa_5,PPPPGSSSSSTTTTNNQQQ,A
5,hpa_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

HYDROPATHY = {
    "A": 1.8, "C": 2.5, "D": -3.5, "E": -3.5, "F": 2.8,
    "G": -0.4, "H": -3.2, "I": 4.5, "K": -3.9, "L": 3.8,
    "M": 1.9, "N": -3.5, "P": -1.6, "Q": -3.5, "R": -4.5,
    "S": -0.8, "T": -0.7, "V": 4.2, "W": -0.9, "Y": -1.3,
}

POLARITY = {
    "A": 8.1, "C": 5.5, "D": 13.0, "E": 12.3, "F": 5.2,
    "G": 9.0, "H": 10.4, "I": 5.2, "K": 11.3, "L": 4.9,
    "M": 5.7, "N": 11.6, "P": 8.0, "Q": 10.5, "R": 10.5,
    "S": 9.2, "T": 8.6, "V": 5.9, "W": 5.4, "Y": 6.2,
}

HYDROPHOBIC = set("AVLIMFWCY")
HYDROPHILIC = set("RNDQEHKST")
POLAR = set("STNQCYWHKRDE")
NONPOLAR = set("AVLIMFGP")
AROMATIC = set("FWYH")


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def windows(seq: str, size: int):
    if len(seq) < size:
        return []
    return [seq[i:i+size] for i in range(len(seq) - size + 1)]


def scale_values(seq: str, scale: dict):
    return [scale[aa] for aa in seq]


def scale_mean(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    return float(np.mean(scale_values(seq, scale)))


def scale_std(seq: str, scale: dict) -> float:
    if len(seq) == 0:
        return np.nan
    return float(np.std(scale_values(seq, scale), ddof=0))


def fraction_from_group(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def profile_stats(values):
    if len(values) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "amplitude": np.nan,
            "start_end_diff": np.nan,
        }
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=0)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "amplitude": float(np.max(values) - np.min(values)),
        "start_end_diff": float(values[-1] - values[0]),
    }


def fraction_above_threshold(values, threshold: float) -> float:
    if len(values) == 0:
        return np.nan
    return float(np.mean(np.array(values) > threshold))


def terminal_segment(seq: str, side: str = "N", window: int = 10) -> str:
    if side == "N":
        return seq[:window]
    if side == "C":
        return seq[-window:]
    raise ValueError("side must be 'N' or 'C'")


def local_contrast_profile(seq: str, window: int = 5):
    ws = windows(seq, window)
    if len(ws) == 0:
        return []
    contrasts = []
    for w in ws:
        hydrophobic_frac = fraction_from_group(w, HYDROPHOBIC)
        polar_frac = fraction_from_group(w, POLAR)
        contrasts.append(abs(hydrophobic_frac - polar_frac))
    return contrasts


def local_amphipathicity_proxy(seq: str, window: int = 5):
    ws = windows(seq, window)
    if len(ws) == 0:
        return []
    values = []
    for w in ws:
        hydro_mean = scale_mean(w, HYDROPATHY)
        polarity_mean = scale_mean(w, POLARITY)
        hydrophobic_frac = fraction_from_group(w, HYDROPHOBIC)
        polar_frac = fraction_from_group(w, POLAR)
        values.append(abs(hydrophobic_frac - polar_frac) * abs(hydro_mean - polarity_mean))
    return values


## Core descriptor function

In [5]:

def hydrophobicity_polarity_amphipathicity_descriptors(seq: str, window_sizes=(5, 7, 9), terminal_window=10) -> dict:
    seq = clean_sequence(seq)

    out = {
        "hpa_length": len(seq),
        "hpa_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    # Global descriptors
    out["hpa_hydropathy_mean"] = scale_mean(seq, HYDROPATHY)
    out["hpa_hydropathy_std"] = scale_std(seq, HYDROPATHY)
    out["hpa_polarity_mean"] = scale_mean(seq, POLARITY)
    out["hpa_polarity_std"] = scale_std(seq, POLARITY)

    out["hpa_hydrophobic_fraction"] = fraction_from_group(seq, HYDROPHOBIC)
    out["hpa_hydrophilic_fraction"] = fraction_from_group(seq, HYDROPHILIC)
    out["hpa_polar_fraction"] = fraction_from_group(seq, POLAR)
    out["hpa_nonpolar_fraction"] = fraction_from_group(seq, NONPOLAR)
    out["hpa_aromatic_fraction"] = fraction_from_group(seq, AROMATIC)

    out["hpa_hydrophobic_hydrophilic_balance"] = out["hpa_hydrophobic_fraction"] - out["hpa_hydrophilic_fraction"]
    out["hpa_polar_nonpolar_balance"] = out["hpa_polar_fraction"] - out["hpa_nonpolar_fraction"]
    out["hpa_global_amphipathicity_proxy"] = abs(out["hpa_hydrophobic_fraction"] - out["hpa_polar_fraction"]) * abs(out["hpa_hydropathy_mean"] - out["hpa_polarity_mean"])

    # Terminal asymmetry
    nterm = terminal_segment(seq, side="N", window=terminal_window)
    cterm = terminal_segment(seq, side="C", window=terminal_window)
    out["hpa_nterm_hydropathy_mean"] = scale_mean(nterm, HYDROPATHY)
    out["hpa_cterm_hydropathy_mean"] = scale_mean(cterm, HYDROPATHY)
    out["hpa_nterm_polarity_mean"] = scale_mean(nterm, POLARITY)
    out["hpa_cterm_polarity_mean"] = scale_mean(cterm, POLARITY)
    out["hpa_terminal_hydropathy_asymmetry"] = out["hpa_nterm_hydropathy_mean"] - out["hpa_cterm_hydropathy_mean"]
    out["hpa_terminal_polarity_asymmetry"] = out["hpa_nterm_polarity_mean"] - out["hpa_cterm_polarity_mean"]

    # Local profiles
    for window_size in window_sizes:
        ws = windows(seq, window_size)

        hydro_profile = [scale_mean(w, HYDROPATHY) for w in ws]
        polarity_profile = [scale_mean(w, POLARITY) for w in ws]
        hydrophobic_frac_profile = [fraction_from_group(w, HYDROPHOBIC) for w in ws]
        polar_frac_profile = [fraction_from_group(w, POLAR) for w in ws]
        contrast_profile = local_contrast_profile(seq, window=window_size)
        amphi_profile = local_amphipathicity_proxy(seq, window=window_size)

        profile_map = {
            "hydropathy": hydro_profile,
            "polarity": polarity_profile,
            "hydrophobic_frac": hydrophobic_frac_profile,
            "polar_frac": polar_frac_profile,
            "contrast": contrast_profile,
            "amphipathicity": amphi_profile,
        }

        for name, values in profile_map.items():
            stats = profile_stats(values)
            prefix = f"hpa_w{window_size}_{name}"

            out[f"{prefix}_mean"] = stats["mean"]
            out[f"{prefix}_std"] = stats["std"]
            out[f"{prefix}_min"] = stats["min"]
            out[f"{prefix}_max"] = stats["max"]
            out[f"{prefix}_amplitude"] = stats["amplitude"]
            out[f"{prefix}_start_end_diff"] = stats["start_end_diff"]

        out[f"hpa_w{window_size}_hydrophobic_patch_fraction"] = fraction_above_threshold(hydrophobic_frac_profile, threshold=0.6)
        out[f"hpa_w{window_size}_polar_patch_fraction"] = fraction_above_threshold(polar_frac_profile, threshold=0.6)
        out[f"hpa_w{window_size}_high_amphipathicity_fraction"] = fraction_above_threshold(amphi_profile, threshold=1.0)

    return out


## Functional usage on one sequence

In [6]:

example = hydrophobicity_polarity_amphipathicity_descriptors(
    df_demo.loc[0, "sequence"],
    window_sizes=(5, 7),
    terminal_window=10,
)
list(example.items())[:20]


[('hpa_length', 24),
 ('hpa_valid_residue_count', 24),
 ('hpa_hydropathy_mean', 0.6375000000000001),
 ('hpa_hydropathy_std', 2.95082086037089),
 ('hpa_polarity_mean', 7.295833333333334),
 ('hpa_polarity_std', 2.2080306396324203),
 ('hpa_hydrophobic_fraction', 0.5833333333333334),
 ('hpa_hydrophilic_fraction', 0.375),
 ('hpa_polar_fraction', 0.4583333333333333),
 ('hpa_nonpolar_fraction', 0.5416666666666666),
 ('hpa_aromatic_fraction', 0.25),
 ('hpa_hydrophobic_hydrophilic_balance', 0.20833333333333337),
 ('hpa_polar_nonpolar_balance', -0.08333333333333331),
 ('hpa_global_amphipathicity_proxy', 0.8322916666666671),
 ('hpa_nterm_hydropathy_mean', 1.47),
 ('hpa_cterm_hydropathy_mean', -0.8),
 ('hpa_nterm_polarity_mean', 6.63),
 ('hpa_cterm_polarity_mean', 8.43),
 ('hpa_terminal_hydropathy_asymmetry', 2.27),
 ('hpa_terminal_polarity_asymmetry', -1.7999999999999998)]

## Apply descriptors to the full dataset

In [7]:

df_hpa = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: hydrophobicity_polarity_amphipathicity_descriptors(
                x,
                window_sizes=(5, 7, 9),
                terminal_window=10,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_hpa.head()


,sequence_id,sequence,label,hpa_length,hpa_valid_residue_count,hpa_hydropathy_mean,hpa_hydropathy_std,hpa_polarity_mean,hpa_polarity_std,hpa_hydrophobic_fraction,...,hpa_w9_contrast_start_end_diff,hpa_w9_amphipathicity_mean,hpa_w9_amphipathicity_std,hpa_w9_amphipathicity_min,hpa_w9_amphipathicity_max,hpa_w9_amphipathicity_amplitude,hpa_w9_amphipathicity_start_end_diff,hpa_w9_hydrophobic_patch_fraction,hpa_w9_polar_patch_fraction,hpa_w9_high_amphipathicity_fraction
0,hpa_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.637500,2.950821e+00,7.295833,2.208031,0.583333,...,-0.111111,1.303086,0.521307,0.000000,2.069136,2.069136,-0.230864,0.625,0.000000,0.687500
1,hpa_2,GGGGGGGGGGGGGGG,B,15.0,15.0,-0.400000,1.110223e-16,9.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000
2,hpa_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,-4.033333,4.422166e-01,11.433333,1.009950,0.000000,...,0.000000,15.373333,0.306320,15.066667,15.866667,0.800000,0.800000,0.000,1.000000,1.000000
3,hpa_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,-0.490000,2.911340e+00,8.325000,2.622380,0.450000,...,0.333333,1.851029,1.463592,0.000000,4.597531,4.597531,3.346914,0.000,0.250000,0.666667
4,hpa_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,-1.636842,1.167638e+00,9.268421,1.117823,0.000000,...,0.555556,8.943996,2.532700,4.335802,12.155556,7.819753,7.819753,0.000,0.818182,1.000000


## Inspect descriptor columns

In [8]:

hpa_cols = [c for c in df_hpa.columns if c.startswith("hpa_") and c not in {"hpa_length", "hpa_valid_residue_count"}]
len(hpa_cols), hpa_cols[:18]


(135,
 ['hpa_hydropathy_mean',
  'hpa_hydropathy_std',
  'hpa_polarity_mean',
  'hpa_polarity_std',
  'hpa_hydrophobic_fraction',
  'hpa_hydrophilic_fraction',
  'hpa_polar_fraction',
  'hpa_nonpolar_fraction',
  'hpa_aromatic_fraction',
  'hpa_hydrophobic_hydrophilic_balance',
  'hpa_polar_nonpolar_balance',
  'hpa_global_amphipathicity_proxy',
  'hpa_nterm_hydropathy_mean',
  'hpa_cterm_hydropathy_mean',
  'hpa_nterm_polarity_mean',
  'hpa_cterm_polarity_mean',
  'hpa_terminal_hydropathy_asymmetry',
  'hpa_terminal_polarity_asymmetry'])

In [9]:

df_hpa[
    [
        "sequence_id",
        "hpa_hydropathy_mean",
        "hpa_polarity_mean",
        "hpa_hydrophobic_fraction",
        "hpa_polar_fraction",
        "hpa_global_amphipathicity_proxy",
        "hpa_w5_amphipathicity_mean",
        "hpa_w7_contrast_amplitude",
        "hpa_w9_high_amphipathicity_fraction",
    ]
]


,sequence_id,hpa_hydropathy_mean,hpa_polarity_mean,hpa_hydrophobic_fraction,hpa_polar_fraction,hpa_global_amphipathicity_proxy,hpa_w5_amphipathicity_mean,hpa_w7_contrast_amplitude,hpa_w9_high_amphipathicity_fraction
0,hpa_1,0.637500,7.295833,0.583333,0.458333,0.832292,1.303000,0.714286,0.687500
1,hpa_2,-0.400000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,hpa_3,-4.033333,11.433333,0.000000,1.000000,15.466667,15.441429,0.000000,1.000000
3,hpa_4,-0.490000,8.325000,0.450000,0.600000,1.322250,2.759000,0.571429,0.666667
4,hpa_5,-1.636842,9.268421,0.000000,0.736842,8.035457,8.718667,0.714286,1.000000
5,hpa_6,-0.814286,8.919048,0.285714,0.571429,2.780952,4.794588,0.714286,1.000000


## Dataset-level summary

In [10]:

hpa_summary = (
    df_hpa[hpa_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

hpa_summary.head(15)


,descriptor,mean_value
0,hpa_w5_polarity_max,10.443333
1,hpa_w7_polarity_max,10.100000
2,hpa_w9_polarity_max,9.672222
3,hpa_cterm_polarity_mean,9.395000
4,hpa_w5_amphipathicity_max,9.386667
5,hpa_w5_polarity_mean,9.093767
6,hpa_w7_polarity_mean,9.063435
7,hpa_polarity_mean,9.040273
8,hpa_w9_polarity_mean,9.026743
9,hpa_nterm_polarity_mean,8.750000


## Sanity checks

In [11]:

assert "hpa_hydropathy_mean" in df_hpa.columns
assert "hpa_polarity_mean" in df_hpa.columns
assert "hpa_global_amphipathicity_proxy" in df_hpa.columns
assert "hpa_w5_amphipathicity_mean" in df_hpa.columns
assert "hpa_w7_contrast_amplitude" in df_hpa.columns
assert "hpa_w9_high_amphipathicity_fraction" in df_hpa.columns
assert df_hpa["hpa_length"].min() > 0

print(f"Number of hydrophobicity/polarity/amphipathicity descriptor columns: {len(hpa_cols)}")
print("Hydrophobicity/polarity/amphipathicity descriptor checks passed.")


Number of hydrophobicity/polarity/amphipathicity descriptor columns: 135
Hydrophobicity/polarity/amphipathicity descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class HydrophobicityPolarityAmphipathicityDescriptors:
    """Example class-style implementation for later migration into Roxy."""

    def __init__(self, window_sizes=(5, 7, 9), terminal_window=10):
        self.window_sizes = tuple(window_sizes)
        self.terminal_window = int(terminal_window)

    def transform_sequence(self, seq: str) -> dict:
        return hydrophobicity_polarity_amphipathicity_descriptors(
            seq,
            window_sizes=self.window_sizes,
            terminal_window=self.terminal_window,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


hpa_transformer = HydrophobicityPolarityAmphipathicityDescriptors(window_sizes=(5, 7, 9), terminal_window=10)
hpa_matrix = hpa_transformer.transform(df_demo["sequence"].tolist())
hpa_matrix.head()


,hpa_length,hpa_valid_residue_count,hpa_hydropathy_mean,hpa_hydropathy_std,hpa_polarity_mean,hpa_polarity_std,hpa_hydrophobic_fraction,hpa_hydrophilic_fraction,hpa_polar_fraction,hpa_nonpolar_fraction,...,hpa_w9_contrast_start_end_diff,hpa_w9_amphipathicity_mean,hpa_w9_amphipathicity_std,hpa_w9_amphipathicity_min,hpa_w9_amphipathicity_max,hpa_w9_amphipathicity_amplitude,hpa_w9_amphipathicity_start_end_diff,hpa_w9_hydrophobic_patch_fraction,hpa_w9_polar_patch_fraction,hpa_w9_high_amphipathicity_fraction
0,24,24,0.637500,2.950821e+00,7.295833,2.208031,0.583333,0.375000,0.458333,0.541667,...,-0.111111,1.303086,0.521307,0.000000,2.069136,2.069136,-0.230864,0.625,0.000000,0.687500
1,15,15,-0.400000,1.110223e-16,9.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000
2,18,18,-4.033333,4.422166e-01,11.433333,1.009950,0.000000,1.000000,1.000000,0.000000,...,0.000000,15.373333,0.306320,15.066667,15.866667,0.800000,0.800000,0.000,1.000000,1.000000
3,20,20,-0.490000,2.911340e+00,8.325000,2.622380,0.450000,0.450000,0.600000,0.400000,...,0.333333,1.851029,1.463592,0.000000,4.597531,4.597531,3.346914,0.000,0.250000,0.666667
4,19,19,-1.636842,1.167638e+00,9.268421,1.117823,0.000000,0.736842,0.736842,0.263158,...,0.555556,8.943996,2.532700,4.335802,12.155556,7.819753,7.819753,0.000,0.818182,1.000000


## Merge transformer output back to the dataset

In [13]:

df_hpa_class = pd.concat([df_demo, hpa_matrix], axis=1)
df_hpa_class.head()


,sequence_id,sequence,label,hpa_length,hpa_valid_residue_count,hpa_hydropathy_mean,hpa_hydropathy_std,hpa_polarity_mean,hpa_polarity_std,hpa_hydrophobic_fraction,...,hpa_w9_contrast_start_end_diff,hpa_w9_amphipathicity_mean,hpa_w9_amphipathicity_std,hpa_w9_amphipathicity_min,hpa_w9_amphipathicity_max,hpa_w9_amphipathicity_amplitude,hpa_w9_amphipathicity_start_end_diff,hpa_w9_hydrophobic_patch_fraction,hpa_w9_polar_patch_fraction,hpa_w9_high_amphipathicity_fraction
0,hpa_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.637500,2.950821e+00,7.295833,2.208031,0.583333,...,-0.111111,1.303086,0.521307,0.000000,2.069136,2.069136,-0.230864,0.625,0.000000,0.687500
1,hpa_2,GGGGGGGGGGGGGGG,B,15,15,-0.400000,1.110223e-16,9.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000000,0.000000
2,hpa_3,KRRKRRKRRKRRDDDDEE,A,18,18,-4.033333,4.422166e-01,11.433333,1.009950,0.000000,...,0.000000,15.373333,0.306320,15.066667,15.866667,0.800000,0.800000,0.000,1.000000,1.000000
3,hpa_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,-0.490000,2.911340e+00,8.325000,2.622380,0.450000,...,0.333333,1.851029,1.463592,0.000000,4.597531,4.597531,3.346914,0.000,0.250000,0.666667
4,hpa_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,-1.636842,1.167638e+00,9.268421,1.117823,0.000000,...,0.555556,8.943996,2.532700,4.335802,12.155556,7.819753,7.819753,0.000,0.818182,1.000000



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move residue scales into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/hydropathy.py` or a dedicated `physchem_profiles.py`
- expose a class such as `HydrophobicityPolarityAmphipathicityDescriptors`
- allow configurable:
  - window sizes
  - terminal window size
  - local patch thresholds
  - selected profile summaries
- add tests for:
  - empty sequences
  - strongly hydrophobic sequences
  - strongly polar sequences
  - mixed amphipathic-like sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_hpa.to_csv("demo_hydrophobicity_polarity_amphipathicity_descriptors.csv", index=False)
